In [1]:
import pandas as pd
import matplotlib.pyplot as plt

/tmp/ipykernel_24948/1492400551.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [2]:
DATA_PATH = '../../data'
SBCR_RESULTS = '../../sbcr'

SBCR_DATA = f"{SBCR_RESULTS}/tunning_dataset2_TypeScript/RESULTS/results_tunning_dataset2_TypeScript_combination_rq1_dataset2_TypeScript_seed-3022024.xlsx"

## SBCR tunning on dataset2_TypeScript

In [3]:
df = pd.read_excel(SBCR_DATA, engine='openpyxl')
df

,chunk_id,fitness,solution_sim_lcs,status,neighbor_timeout,local_search_timeout,ils_timeout,maximum_neighbors,max_iterations_without_improvement,time_seconds
0,53846-2,0.931057,0.888889,ok,3,5,10,1,5,0.018153
1,53846-2,0.931057,0.888889,ok,3,5,20,1,5,0.025091
2,53846-2,0.930880,1.000000,ok,3,5,30,1,5,0.011177
3,53846-2,0.930898,0.863636,ok,3,5,40,1,5,0.008367
4,53846-2,0.931057,0.888889,ok,3,5,10,3,5,0.064721
...,...,...,...,...,...,...,...,...,...,...
7995,16741-2,0.831615,0.660465,ok,3,5,40,7,20,24.345212
7996,16741-2,0.831615,0.660465,ok,3,5,10,9,20,13.530417
7997,16741-2,0.831615,0.660465,ok,3,5,20,9,20,24.839737
7998,16741-2,0.831615,0.660465,ok,3,5,30,9,20,35.285705


In [4]:
df['configuration'] = list(zip(df['maximum_neighbors'], df['max_iterations_without_improvement'], df['ils_timeout']))
df = df.drop_duplicates(subset=['chunk_id', 'configuration'])
df

,chunk_id,fitness,solution_sim_lcs,status,neighbor_timeout,local_search_timeout,ils_timeout,maximum_neighbors,max_iterations_without_improvement,time_seconds,configuration
0,53846-2,0.931057,0.888889,ok,3,5,10,1,5,0.018153,"(1, 5, 10)"
1,53846-2,0.931057,0.888889,ok,3,5,20,1,5,0.025091,"(1, 5, 20)"
2,53846-2,0.930880,1.000000,ok,3,5,30,1,5,0.011177,"(1, 5, 30)"
3,53846-2,0.930898,0.863636,ok,3,5,40,1,5,0.008367,"(1, 5, 40)"
4,53846-2,0.931057,0.888889,ok,3,5,10,3,5,0.064721,"(3, 5, 10)"
...,...,...,...,...,...,...,...,...,...,...,...
7995,16741-2,0.831615,0.660465,ok,3,5,40,7,20,24.345212,"(7, 20, 40)"
7996,16741-2,0.831615,0.660465,ok,3,5,10,9,20,13.530417,"(9, 20, 10)"
7997,16741-2,0.831615,0.660465,ok,3,5,20,9,20,24.839737,"(9, 20, 20)"
7998,16741-2,0.831615,0.660465,ok,3,5,30,9,20,35.285705,"(9, 20, 30)"


In [5]:
df=df.dropna()
df

,chunk_id,fitness,solution_sim_lcs,status,neighbor_timeout,local_search_timeout,ils_timeout,maximum_neighbors,max_iterations_without_improvement,time_seconds,configuration
0,53846-2,0.931057,0.888889,ok,3,5,10,1,5,0.018153,"(1, 5, 10)"
1,53846-2,0.931057,0.888889,ok,3,5,20,1,5,0.025091,"(1, 5, 20)"
2,53846-2,0.930880,1.000000,ok,3,5,30,1,5,0.011177,"(1, 5, 30)"
3,53846-2,0.930898,0.863636,ok,3,5,40,1,5,0.008367,"(1, 5, 40)"
4,53846-2,0.931057,0.888889,ok,3,5,10,3,5,0.064721,"(3, 5, 10)"
...,...,...,...,...,...,...,...,...,...,...,...
7995,16741-2,0.831615,0.660465,ok,3,5,40,7,20,24.345212,"(7, 20, 40)"
7996,16741-2,0.831615,0.660465,ok,3,5,10,9,20,13.530417,"(9, 20, 10)"
7997,16741-2,0.831615,0.660465,ok,3,5,20,9,20,24.839737,"(9, 20, 20)"
7998,16741-2,0.831615,0.660465,ok,3,5,30,9,20,35.285705,"(9, 20, 30)"


In [6]:
df = df[df['status']=='ok']
print(len(df))

8000


#### Rank the parameters by number of victories for each instance

In [7]:
df['configuration'] = list(zip(df['maximum_neighbors'], df['max_iterations_without_improvement'], df['ils_timeout']))

# find the top 1 configuration for each chunk
max_scores = df.groupby('chunk_id')['solution_sim_lcs'].transform('max')
best_configs = df[df['solution_sim_lcs'] == max_scores]

# calculate the number of vicories and average similarity for chunks where the configuration is top 1
victory_count = best_configs.groupby('configuration').agg(
    victories=('configuration', 'count'),
    mean_solution_sim_lcs=('solution_sim_lcs', 'mean')  # avg similarity for top1 chunks
).reset_index()

# calculate average similarity overall for each configuration
mean_similarity_overall = df.groupby('configuration').agg(
    solution_sim_lcs=('solution_sim_lcs','mean'),
    mean_time=('time_seconds','mean')
).reset_index()
mean_similarity_overall.columns = ['configuration', 'mean_solution_sim_lcs_overall', 'mean_time_overall']

result = victory_count.merge(mean_similarity_overall, on='configuration', how='left')
result_sorted = result.sort_values(by='victories', ascending=False)
result_sorted[['configuration', 'victories', 'mean_solution_sim_lcs', 'mean_solution_sim_lcs_overall', 'mean_time_overall']].head(10)

,configuration,victories,mean_solution_sim_lcs,mean_solution_sim_lcs_overall,mean_time_overall
32,"(5, 5, 10)",26,0.906955,0.778860,1.607231
25,"(3, 15, 20)",24,0.933299,0.780746,1.258538
55,"(7, 10, 40)",24,0.892328,0.776476,7.359945
23,"(3, 10, 40)",24,0.920043,0.779991,1.430812
36,"(5, 10, 10)",24,0.943592,0.781629,2.282917
39,"(5, 10, 40)",23,0.926550,0.775792,3.384434
57,"(7, 15, 20)",23,0.922845,0.788335,7.866650
67,"(9, 5, 40)",23,0.926365,0.795158,8.021936
68,"(9, 10, 10)",22,0.932021,0.796260,7.365880
51,"(7, 5, 40)",22,0.936349,0.788717,4.807771


#### Rank the parameters by the ascending value of the sum of ranks (for each instance)

In [8]:
# Sort the configurations by chunk_id and solution_sim_lcs (score)
df['rank'] = df.groupby('chunk_id')['solution_sim_lcs'].rank(method='min', ascending=False)

# Sum the ranks for each configuration
config_ranking = df.groupby('configuration').agg(
    rank_sum=('rank', 'sum'),
    rank_avg=('rank', 'mean'),
    mean_similarity=('solution_sim_lcs', 'mean'),
    min_similarity=('solution_sim_lcs', 'min'),
    max_similarity=('solution_sim_lcs', 'max'),
    median_similarity=('solution_sim_lcs', 'median'),
    mean_time=('time_seconds', 'mean')
).reset_index()

# Classify the configurations based on the sum of ranks (smallest sum is better)
config_ranking['overall_rank'] = config_ranking['rank_sum'].rank(method='min')

#sort by overall ranking
config_ranking_sorted = config_ranking.sort_values(by='overall_rank')

config_ranking_sorted[['configuration', 'rank_sum', 'rank_avg', 'overall_rank', 'mean_similarity','min_similarity', 'max_similarity', 'median_similarity']]

,configuration,rank_sum,rank_avg,overall_rank,mean_similarity,min_similarity,max_similarity,median_similarity
79,"(9, 20, 40)",1378.0,13.78,1.0,0.790527,0.072787,1.0,0.871345
72,"(9, 15, 10)",1449.0,14.49,2.0,0.802431,0.077482,1.0,0.871345
56,"(7, 15, 10)",1455.0,14.55,3.0,0.796977,0.077482,1.0,0.871345
63,"(7, 20, 40)",1468.0,14.68,4.0,0.793081,0.072787,1.0,0.871345
74,"(9, 15, 30)",1469.0,14.69,5.0,0.790042,0.077482,1.0,0.871345
...,...,...,...,...,...,...,...,...
7,"(1, 10, 40)",3760.0,37.60,76.0,0.768256,0.077108,1.0,0.828570
0,"(1, 5, 10)",3788.0,37.88,77.0,0.771287,0.079208,1.0,0.825263
2,"(1, 5, 30)",3816.0,38.16,78.0,0.784222,0.088643,1.0,0.831560
3,"(1, 5, 40)",3855.0,38.55,79.0,0.771967,0.088154,1.0,0.810970


In [9]:
result = config_ranking_sorted[['configuration', 'rank_avg', 'mean_similarity', 'mean_time']].merge(result_sorted[['victories', 'configuration']],on='configuration' ,  how='outer')
result.loc[result['victories'].isnull(),'victories'] = 0
# result.to_csv('result.csv')
result

,configuration,rank_avg,mean_similarity,mean_time,victories
0,"(1, 5, 10)",37.88,0.771287,0.436618,15
1,"(1, 5, 20)",39.62,0.763188,0.382006,12
2,"(1, 5, 30)",38.16,0.784222,0.571407,14
3,"(1, 5, 40)",38.55,0.771967,0.538750,20
4,"(1, 10, 10)",30.22,0.784758,0.357926,16
...,...,...,...,...,...
75,"(9, 15, 40)",16.02,0.790213,16.342954,19
76,"(9, 20, 10)",15.44,0.782957,8.804881,17
77,"(9, 20, 20)",15.06,0.782745,12.504112,18
78,"(9, 20, 30)",15.24,0.798353,17.190650,18


In [10]:
result.sort_values('mean_similarity', ascending=False)

,configuration,rank_avg,mean_similarity,mean_time,victories
72,"(9, 15, 10)",14.49,0.802431,8.160524,20
62,"(7, 20, 30)",16.77,0.801359,12.114661,20
78,"(9, 20, 30)",15.24,0.798353,17.190650,18
35,"(5, 5, 40)",22.05,0.797329,2.261596,21
56,"(7, 15, 10)",14.55,0.796977,5.896768,22
...,...,...,...,...,...
9,"(1, 15, 20)",34.07,0.767679,0.755770,18
21,"(3, 10, 20)",25.56,0.767488,1.448233,20
8,"(1, 15, 10)",33.48,0.766083,0.420152,16
1,"(1, 5, 20)",39.62,0.763188,0.382006,12


In [11]:
result.sort_values('victories', ascending=False)

,configuration,rank_avg,mean_similarity,mean_time,victories
32,"(5, 5, 10)",23.76,0.778860,1.607231,26
25,"(3, 15, 20)",24.03,0.780746,1.258538,24
55,"(7, 10, 40)",18.23,0.776476,7.359945,24
23,"(3, 10, 40)",24.18,0.779991,1.430812,24
36,"(5, 10, 10)",22.15,0.781629,2.282917,24
...,...,...,...,...,...
7,"(1, 10, 40)",37.60,0.768256,0.689088,16
4,"(1, 10, 10)",30.22,0.784758,0.357926,16
0,"(1, 5, 10)",37.88,0.771287,0.436618,15
2,"(1, 5, 30)",38.16,0.784222,0.571407,14


In [12]:
result.sort_values('rank_avg')

,configuration,rank_avg,mean_similarity,mean_time,victories
79,"(9, 20, 40)",13.78,0.790527,19.365339,19
72,"(9, 15, 10)",14.49,0.802431,8.160524,20
56,"(7, 15, 10)",14.55,0.796977,5.896768,22
63,"(7, 20, 40)",14.68,0.793081,11.876048,18
74,"(9, 15, 30)",14.69,0.790042,14.853463,19
...,...,...,...,...,...
7,"(1, 10, 40)",37.60,0.768256,0.689088,16
0,"(1, 5, 10)",37.88,0.771287,0.436618,15
2,"(1, 5, 30)",38.16,0.784222,0.571407,14
3,"(1, 5, 40)",38.55,0.771967,0.538750,20


In [13]:
similarities = df.groupby('chunk_id').agg(
    max_similarity=('solution_sim_lcs', 'max')
).reset_index()
similarities

,chunk_id,max_similarity
0,10503-1,1.000000
1,11459-1,0.588235
2,11651-1,0.912387
3,11972-1,1.000000
4,13188-1,0.866450
...,...,...
95,8586-3,1.000000
96,8588-1,0.945412
97,8730-3,1.000000
98,9502-1,0.823188


In [14]:
target = similarities['max_similarity'].mean()
print(target)

0.9049305773010167


In [15]:
df['target_diff'] = target - df['solution_sim_lcs']
df

,chunk_id,fitness,solution_sim_lcs,status,neighbor_timeout,local_search_timeout,ils_timeout,maximum_neighbors,max_iterations_without_improvement,time_seconds,configuration,rank,target_diff
0,53846-2,0.931057,0.888889,ok,3,5,10,1,5,0.018153,"(1, 5, 10)",6.0,0.016042
1,53846-2,0.931057,0.888889,ok,3,5,20,1,5,0.025091,"(1, 5, 20)",6.0,0.016042
2,53846-2,0.930880,1.000000,ok,3,5,30,1,5,0.011177,"(1, 5, 30)",1.0,-0.095069
3,53846-2,0.930898,0.863636,ok,3,5,40,1,5,0.008367,"(1, 5, 40)",77.0,0.041294
4,53846-2,0.931057,0.888889,ok,3,5,10,3,5,0.064721,"(3, 5, 10)",6.0,0.016042
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7995,16741-2,0.831615,0.660465,ok,3,5,40,7,20,24.345212,"(7, 20, 40)",12.0,0.244465
7996,16741-2,0.831615,0.660465,ok,3,5,10,9,20,13.530417,"(9, 20, 10)",12.0,0.244465
7997,16741-2,0.831615,0.660465,ok,3,5,20,9,20,24.839737,"(9, 20, 20)",12.0,0.244465
7998,16741-2,0.831615,0.660465,ok,3,5,30,9,20,35.285705,"(9, 20, 30)",12.0,0.244465


In [16]:
len(df[df['solution_sim_lcs']==1]['chunk_id'].unique())

56

In [17]:
df.sort_values('target_diff')

,chunk_id,fitness,solution_sim_lcs,status,neighbor_timeout,local_search_timeout,ils_timeout,maximum_neighbors,max_iterations_without_improvement,time_seconds,configuration,rank,target_diff
3999,4801-1,0.809524,1.000000,ok,3,5,40,9,20,43.785947,"(9, 20, 40)",1.0,-0.095069
5647,30561-3,0.832012,1.000000,ok,3,5,40,3,15,0.177278,"(3, 15, 40)",1.0,-0.095069
5646,30561-3,0.831582,1.000000,ok,3,5,30,3,15,0.089395,"(3, 15, 30)",1.0,-0.095069
5645,30561-3,0.831582,1.000000,ok,3,5,20,3,15,0.115047,"(3, 15, 20)",1.0,-0.095069
5644,30561-3,0.832012,1.000000,ok,3,5,10,3,15,0.134558,"(3, 15, 10)",1.0,-0.095069
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7670,10503-1,0.534305,0.072787,ok,3,5,30,5,20,13.286010,"(5, 20, 30)",54.0,0.832143
7657,10503-1,0.534305,0.072787,ok,3,5,20,9,15,24.238897,"(9, 15, 20)",54.0,0.832143
7625,10503-1,0.534305,0.072787,ok,3,5,20,3,10,2.640022,"(3, 10, 20)",54.0,0.832143
1852,42332-1,0.489176,0.071979,ok,3,5,10,7,5,0.271398,"(7, 5, 10)",79.0,0.832951


#### Chunks with zero similarity to the resolution

In [18]:
df

,chunk_id,fitness,solution_sim_lcs,status,neighbor_timeout,local_search_timeout,ils_timeout,maximum_neighbors,max_iterations_without_improvement,time_seconds,configuration,rank,target_diff
0,53846-2,0.931057,0.888889,ok,3,5,10,1,5,0.018153,"(1, 5, 10)",6.0,0.016042
1,53846-2,0.931057,0.888889,ok,3,5,20,1,5,0.025091,"(1, 5, 20)",6.0,0.016042
2,53846-2,0.930880,1.000000,ok,3,5,30,1,5,0.011177,"(1, 5, 30)",1.0,-0.095069
3,53846-2,0.930898,0.863636,ok,3,5,40,1,5,0.008367,"(1, 5, 40)",77.0,0.041294
4,53846-2,0.931057,0.888889,ok,3,5,10,3,5,0.064721,"(3, 5, 10)",6.0,0.016042
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7995,16741-2,0.831615,0.660465,ok,3,5,40,7,20,24.345212,"(7, 20, 40)",12.0,0.244465
7996,16741-2,0.831615,0.660465,ok,3,5,10,9,20,13.530417,"(9, 20, 10)",12.0,0.244465
7997,16741-2,0.831615,0.660465,ok,3,5,20,9,20,24.839737,"(9, 20, 20)",12.0,0.244465
7998,16741-2,0.831615,0.660465,ok,3,5,30,9,20,35.285705,"(9, 20, 30)",12.0,0.244465


In [19]:
data = []
chunks_zero_sim = df[df['solution_sim_lcs']==0]['chunk_id'].unique()
for chunk_zero_sim in chunks_zero_sim:
    df_chunk = df[(df['chunk_id']==chunk_zero_sim) & (df['solution_sim_lcs']==0)]
    chunk_data = df[(df['chunk_id']==chunk_zero_sim)].iloc[0]
    data.append([chunk_zero_sim, len(df_chunk), (len(df_chunk)/27)*100, chunk_data['chunk_size_char'], chunk_data['solution_size_char'], chunk_data['v1_size_char'], chunk_data['v2_size_char']])
zero_sim = pd.DataFrame(data, columns=['chunk_id', 'zero_sim_configs', '%', 'chunk_size_char','solution_size_char', 'v1_size_char', 'v2_size_char'])
zero_sim

,chunk_id,zero_sim_configs,%,chunk_size_char,solution_size_char,v1_size_char,v2_size_char


In [20]:
len(zero_sim[zero_sim['%']==100])

0